In [60]:
import pandas as pd
import numpy as np

shipment=pd.read_csv('shpiment_cleaned2.csv')
shipment_leg=pd.read_csv('shipment_leg_cleaned.csv')
ports=pd.read_csv('ports.csv')
carriers=pd.read_csv('carriers.csv')
carbon_log=pd.read_csv('carbon_logs.csv')
telemetry=pd.read_csv('telemetry.csv')
tariffs_customs=pd.read_csv('tariffs_customs.csv')

In [61]:
valid_ports = set(ports["PortID"].astype(str).str.strip())

shipment_leg["InvalidOriginPortFlag"] = ~shipment_leg["OriginPortID"].isin(valid_ports)
shipment_leg["InvalidDestinationPortFlag"] = ~shipment_leg["DestinationPortID"].isin(valid_ports)

In [62]:
valid_carriers = set(carriers["CarrierID"].astype(str).str.strip())

shipment_leg["InvalidCarrierFlag"] = ~shipment_leg["CarrierID"].isin(valid_carriers)

In [63]:
#shipment_leg.to_csv('shipment_leg_cleaned2.csv', index=False)

In [64]:
ports.info()
ports.head(20)



<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   PortID           15 non-null     str    
 1   PortName         15 non-null     str    
 2   Country          15 non-null     str    
 3   HandlingFee_USD  15 non-null     int64  
 4   Capacity_TEU     15 non-null     int64  
 5   Latitude         15 non-null     float64
 6   Longitude        15 non-null     float64
dtypes: float64(2), int64(2), str(3)
memory usage: 1.4 KB


,PortID,PortName,Country,HandlingFee_USD,Capacity_TEU,Latitude,Longitude
0,P01,Port of Shanghai,China,850,45000000,31.2304,121.4737
1,P02,Port of Shenzhen,China,800,30000000,22.5431,114.0579
2,P03,Port of Los Angeles,United States,1500,10000000,33.7432,-118.2673
3,P04,Port of Long Beach,United States,1450,9000000,33.7701,-118.1937
4,P05,Port of Rotterdam,Netherlands,1100,15000000,51.9244,4.4777
5,P06,Port of Hamburg,Germany,1200,9000000,53.5511,9.9937
6,P07,Port of Da Nang,Vietnam,550,2000000,16.0544,108.2022
7,P08,Port of Ho Chi Minh,Vietnam,600,7000000,10.8231,106.6297
8,P09,Port of Singapore,Singapore,750,37000000,1.3521,103.8198
9,P10,Port of Tokyo,Japan,1000,5000000,35.6762,139.6503


In [65]:
carriers.info()
carriers.head(20)

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CarrierID          10 non-null     str    
 1   CarrierName        10 non-null     str    
 2   Mode               10 non-null     str    
 3   SLA_Days           10 non-null     int64  
 4   LatePenalty_USD    10 non-null     int64  
 5   CO2_Factor         10 non-null     float64
 6   ReportedESGRating  10 non-null     str    
dtypes: float64(1), int64(2), str(4)
memory usage: 953.0 bytes


,CarrierID,CarrierName,Mode,SLA_Days,LatePenalty_USD,CO2_Factor,ReportedESGRating
0,C01,Apex Ocean Transport,Ocean,15,5000,0.012,A
1,C02,Oceanic Swift Lines,Ocean,12,8000,0.014,B
2,C03,Pacific Horizon,Ocean,18,3000,0.011,A
3,C04,FrostWing Logistics,Air,3,15000,0.520,A+
4,C05,EcoTransit Express,Road,5,2500,0.080,AAA
5,C06,Global Iron Rail,Rail,8,4000,0.025,B
6,C07,Polar Air Cargo,Air,2,20000,0.550,A
7,C08,Trans-America Freight,Road,6,2000,0.090,C
8,C09,QuickLoad LastMile,Road,2,1000,0.120,B
9,C10,Vanguard Marine,Ocean,20,2000,0.010,A


In [66]:
ports = pd.read_csv("ports.csv", dtype={"PortID": "str"})

In [67]:
text_columns = ["PortID", "PortName", "Country"]

for col in text_columns:
    ports[col] = ports[col].astype(str).str.strip()

In [68]:
ports["HandlingFee_USD"] = pd.to_numeric(ports["HandlingFee_USD"], errors="coerce")
ports["Capacity_TEU"] = pd.to_numeric(ports["Capacity_TEU"], errors="coerce")
ports["Latitude"] = pd.to_numeric(ports["Latitude"], errors="coerce")
ports["Longitude"] = pd.to_numeric(ports["Longitude"], errors="coerce")

In [69]:
ports["NegativeHandlingFeeFlag"] = ports["HandlingFee_USD"] < 0
ports["NegativeCapacityFlag"] = ports["Capacity_TEU"] < 0
ports["MissingHandlingFeeFlag"] = ports["HandlingFee_USD"].isna()
ports["MissingCapacityFlag"] = ports["Capacity_TEU"].isna()

In [70]:
ports["InvalidLatitudeFlag"] = (
    (ports["Latitude"] < -90) |
    (ports["Latitude"] > 90) |
    ports["Latitude"].isna()
)

ports["InvalidLongitudeFlag"] = (
    (ports["Longitude"] < -180) |
    (ports["Longitude"] > 180) |
    ports["Longitude"].isna()
)

In [71]:
country_to_region = {
    "China": "Asia",
    "Japan": "Asia",
    "Vietnam": "Asia",
    "Singapore": "Asia",
    "United States": "North America",
    "Canada": "North America",
    "Netherlands": "Europe",
    "Germany": "Europe",
    "United Kingdom": "Europe",
    "Belgium": "Europe",
    "Brazil": "South America"
}

ports["Region"] = ports["Country"].map(country_to_region).fillna("Other")

In [72]:
ports["PortFullName"] = (
    ports["PortName"] + ", " + ports["Country"]
)

In [73]:
ports["PortGroupKey"] = (
    ports["PortName"].str.lower().str.strip() +
    "|" +
    ports["Country"].str.lower().str.strip()
)

ports["DuplicatePortGroupFlag"] = (
    ports.duplicated(subset="PortGroupKey", keep=False)
)

In [74]:
print(ports[ports["DuplicatePortGroupFlag"] == True])

   PortID         PortName  Country  HandlingFee_USD  Capacity_TEU  Latitude  \
5     P06  Port of Hamburg  Germany             1200       9000000   53.5511   
10    P11  Port of Hamburg  Germany             1200       9000000   53.5511   

    Longitude  NegativeHandlingFeeFlag  NegativeCapacityFlag  \
5      9.9937                    False                 False   
10     9.9937                    False                 False   

    MissingHandlingFeeFlag  MissingCapacityFlag  InvalidLatitudeFlag  \
5                    False                False                False   
10                   False                False                False   

    InvalidLongitudeFlag  Region              PortFullName  \
5                  False  Europe  Port of Hamburg, Germany   
10                 False  Europe  Port of Hamburg, Germany   

               PortGroupKey  DuplicatePortGroupFlag  
5   port of hamburg|germany                    True  
10  port of hamburg|germany                    True  


In [75]:
ports["MasterPortID"] = ports["PortID"].replace({
    "P11": "P06"
})

In [76]:
port_cluster_map = {
    "P03": "LOS_ANGELES_LONG_BEACH",
    "P04": "LOS_ANGELES_LONG_BEACH",
    "P06": "HAMBURG",
    "P11": "HAMBURG"
}

ports["PortClusterID"] = (
    ports["PortID"]
    .map(port_cluster_map)
    .fillna(ports["PortID"])
)

In [77]:
ports["CapacityCategory"] = pd.cut(
    ports["Capacity_TEU"],
    bins=[0, 5_000_000, 10_000_000, 20_000_000, np.inf],
    labels=["Small", "Medium", "Large", "Mega"]
)

In [78]:
ports["HandlingFeeCategory"] = pd.cut(
    ports["HandlingFee_USD"],
    bins=[0, 800, 1200, np.inf],
    labels=["Low", "Medium", "High"]
)

In [79]:
ports.info()

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   PortID                   15 non-null     str     
 1   PortName                 15 non-null     str     
 2   Country                  15 non-null     str     
 3   HandlingFee_USD          15 non-null     int64   
 4   Capacity_TEU             15 non-null     int64   
 5   Latitude                 15 non-null     float64 
 6   Longitude                15 non-null     float64 
 7   NegativeHandlingFeeFlag  15 non-null     bool    
 8   NegativeCapacityFlag     15 non-null     bool    
 9   MissingHandlingFeeFlag   15 non-null     bool    
 10  MissingCapacityFlag      15 non-null     bool    
 11  InvalidLatitudeFlag      15 non-null     bool    
 12  InvalidLongitudeFlag     15 non-null     bool    
 13  Region                   15 non-null     str     
 14  PortFullName           

In [80]:
ports.to_csv('ports_cleaned.csv', index=False)

In [81]:
text_columns = ["CarrierID", "CarrierName", "Mode", "ReportedESGRating"]

for col in text_columns:
    carriers[col] = carriers[col].astype(str).str.strip()

In [82]:
carriers["SLA_Days"] = pd.to_numeric(carriers["SLA_Days"], errors="coerce")
carriers["LatePenalty_USD"] = pd.to_numeric(carriers["LatePenalty_USD"], errors="coerce")
carriers["CO2_Factor"] = pd.to_numeric(carriers["CO2_Factor"], errors="coerce")

In [83]:
carriers["NegativeSLADaysFlag"] = carriers["SLA_Days"] < 0
carriers["NegativePenaltyFlag"] = carriers["LatePenalty_USD"] < 0
carriers["NegativeCO2FactorFlag"] = carriers["CO2_Factor"] < 0
carriers["MissingSLAFlag"] = carriers["SLA_Days"].isna()
carriers["MissingPenaltyFlag"] = carriers["LatePenalty_USD"].isna()
carriers["MissingCO2FactorFlag"] = carriers["CO2_Factor"].isna()

In [84]:
carriers["Mode"] = carriers["Mode"].str.title()

carriers["Mode"] = carriers["Mode"].replace({
    "Sea": "Ocean",
    "Ocean Freight": "Ocean",
    "Air Freight": "Air",
    "Truck": "Road",
    "Road Freight": "Road",
    "Rail Freight": "Rail"
})

In [85]:
carriers["ReportedESGRating"] = (
    carriers["ReportedESGRating"]
    .str.upper()
    .str.strip()
)

In [86]:
esg_score_map = {
    "AAA": 1,
    "A+": 2,
    "A": 3,
    "B": 4,
    "C": 5
}

carriers["ESGScore"] = carriers["ReportedESGRating"].map(esg_score_map)

In [87]:
print(carriers[carriers["ESGScore"].isna()]["ReportedESGRating"].unique())

<ArrowStringArray>
[]
Length: 0, dtype: str


In [88]:
carriers.info()
carriers.head(20)

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   CarrierID              10 non-null     str    
 1   CarrierName            10 non-null     str    
 2   Mode                   10 non-null     str    
 3   SLA_Days               10 non-null     int64  
 4   LatePenalty_USD        10 non-null     int64  
 5   CO2_Factor             10 non-null     float64
 6   ReportedESGRating      10 non-null     str    
 7   NegativeSLADaysFlag    10 non-null     bool   
 8   NegativePenaltyFlag    10 non-null     bool   
 9   NegativeCO2FactorFlag  10 non-null     bool   
 10  MissingSLAFlag         10 non-null     bool   
 11  MissingPenaltyFlag     10 non-null     bool   
 12  MissingCO2FactorFlag   10 non-null     bool   
 13  ESGScore               10 non-null     int64  
dtypes: bool(6), float64(1), int64(3), str(4)
memory usage: 1.1 KB


,CarrierID,CarrierName,Mode,SLA_Days,LatePenalty_USD,CO2_Factor,ReportedESGRating,NegativeSLADaysFlag,NegativePenaltyFlag,NegativeCO2FactorFlag,MissingSLAFlag,MissingPenaltyFlag,MissingCO2FactorFlag,ESGScore
0,C01,Apex Ocean Transport,Ocean,15,5000,0.012,A,False,False,False,False,False,False,3
1,C02,Oceanic Swift Lines,Ocean,12,8000,0.014,B,False,False,False,False,False,False,4
2,C03,Pacific Horizon,Ocean,18,3000,0.011,A,False,False,False,False,False,False,3
3,C04,FrostWing Logistics,Air,3,15000,0.520,A+,False,False,False,False,False,False,2
4,C05,EcoTransit Express,Road,5,2500,0.080,AAA,False,False,False,False,False,False,1
5,C06,Global Iron Rail,Rail,8,4000,0.025,B,False,False,False,False,False,False,4
6,C07,Polar Air Cargo,Air,2,20000,0.550,A,False,False,False,False,False,False,3
7,C08,Trans-America Freight,Road,6,2000,0.090,C,False,False,False,False,False,False,5
8,C09,QuickLoad LastMile,Road,2,1000,0.120,B,False,False,False,False,False,False,4
9,C10,Vanguard Marine,Ocean,20,2000,0.010,A,False,False,False,False,False,False,3


In [89]:
conditions = [
    carriers["ESGScore"] <= 3,
    carriers["ESGScore"] == 4,
    carriers["ESGScore"] == 5
]

choices = ["High", "Medium", "Low"]

carriers["ESGTier"] = np.select(conditions, choices, default="Unknown")


In [90]:
conditions = [
    carriers["CO2_Factor"] < 0.02,
    carriers["CO2_Factor"] < 0.10
]

choices = [
    "Low Carbon",
    "Medium Carbon"
]

carriers["CO2FactorCategory"] = np.select(
    conditions,
    choices,
    default="High Carbon"
)

In [91]:
carriers["PenaltyCategory"] = pd.cut(
    carriers["LatePenalty_USD"],
    bins=[0, 3000, 8000, np.inf],
    labels=["Low", "Medium", "High"]
)

In [92]:
carriers["SLACategory"] = pd.cut(
    carriers["SLA_Days"],
    bins=[0, 3, 8, np.inf],
    labels=["Fast", "Standard", "Long"]
)

In [93]:
carriers.to_csv('carriers_cleaned.csv')

In [94]:
carriers.info()
carriers.head()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   CarrierID              10 non-null     str     
 1   CarrierName            10 non-null     str     
 2   Mode                   10 non-null     str     
 3   SLA_Days               10 non-null     int64   
 4   LatePenalty_USD        10 non-null     int64   
 5   CO2_Factor             10 non-null     float64 
 6   ReportedESGRating      10 non-null     str     
 7   NegativeSLADaysFlag    10 non-null     bool    
 8   NegativePenaltyFlag    10 non-null     bool    
 9   NegativeCO2FactorFlag  10 non-null     bool    
 10  MissingSLAFlag         10 non-null     bool    
 11  MissingPenaltyFlag     10 non-null     bool    
 12  MissingCO2FactorFlag   10 non-null     bool    
 13  ESGScore               10 non-null     int64   
 14  ESGTier                10 non-null     str     
 15  CO2

,CarrierID,CarrierName,Mode,SLA_Days,LatePenalty_USD,CO2_Factor,ReportedESGRating,NegativeSLADaysFlag,NegativePenaltyFlag,NegativeCO2FactorFlag,MissingSLAFlag,MissingPenaltyFlag,MissingCO2FactorFlag,ESGScore,ESGTier,CO2FactorCategory,PenaltyCategory,SLACategory
0,C01,Apex Ocean Transport,Ocean,15,5000,0.012,A,False,False,False,False,False,False,3,High,Low Carbon,Medium,Long
1,C02,Oceanic Swift Lines,Ocean,12,8000,0.014,B,False,False,False,False,False,False,4,Medium,Low Carbon,Medium,Long
2,C03,Pacific Horizon,Ocean,18,3000,0.011,A,False,False,False,False,False,False,3,High,Low Carbon,Low,Long
3,C04,FrostWing Logistics,Air,3,15000,0.520,A+,False,False,False,False,False,False,2,High,High Carbon,High,Fast
4,C05,EcoTransit Express,Road,5,2500,0.080,AAA,False,False,False,False,False,False,1,High,Medium Carbon,Low,Standard
